# Caching for System Prompts

Reference:
- https://docs.langchain.com/oss/python/integrations/chat/openai#prompt-caching
- https://platform.claude.com/docs/en/build-with-claude/prompt-caching

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv
from pprint import pprint

load_dotenv()

True

In [ ]:
# Load the long static system prompts to test the caching
with open("ai_assistant.txt", "r", encoding="utf-8") as file:
    long_static_system_prompt = file.read()

with open("pride_and_prejudice.txt", "r", encoding="utf-8") as file:
    pride_prejudice_summary = file.read()

In [36]:
# Count tokens using tiktoken (OpenAI's tokenizer)
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1")
tokens = encoding.encode(long_static_system_prompt)
print(f"Character count: {len(long_static_system_prompt)}")
print(f"Token count: {len(tokens)}")

print("--------------------------------")

tokens = encoding.encode(pride_prejudice_summary)
print(f"Character count: {len(pride_prejudice_summary)}")
print(f"Token count: {len(tokens)}")

Character count: 8746
Token count: 1684
--------------------------------
Character count: 16172
Token count: 3093


The model should have more than 1024 tokens now, so the caching should be active here.

## OpenAI

### Langchain

In [ ]:
# Initialize model
model = ChatOpenAI(model="gpt-4o-mini")  # OpenAI auto-caches prompts >1024 tokens

# Long static system prompt
messages = [
    SystemMessage(content=long_static_system_prompt),
    HumanMessage(content="What is the capital of Indonesia and where does it located?")
]

resp = model.invoke(messages)

In [24]:
# Inspect the response with pretty printing
from pprint import pprint

print("=== Full Response ===")
pprint(resp.model_dump())

print("\n=== Content Only ===")
print(resp.content)

print("\n=== Token Usage ===")
pprint(resp.response_metadata.get("token_usage", {}))

=== Full Response ===
{'additional_kwargs': {'refusal': None},
 'content': 'The capital of Indonesia is **Jakarta**. It is located on the '
            'northwest coast of the island of **Java**, which is the most '
            'populous island in Indonesia. Jakarta serves as the political, '
            'economic, and cultural center of the country and is situated in '
            'the Jakarta Special Capital Region. The city is known for its '
            'vibrant culture, diverse population, and significant historical '
            'sites.',
 'id': 'lc_run--019bb7bd-57b8-7573-a5de-f3d3f287dc6b-0',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'finish_reason': 'stop',
                       'id': 'chatcmpl-CxZahZ6dX9VaEWGpsn4OfV3kyOTP9',
                       'logprobs': None,
                       'model_name': 'gpt-4o-mini-2024-07-18',
                       'model_provider': 'openai',
                       'service_tier': 'default',
                       'sy

In [ ]:
# Check cached tokens only
cached = resp.response_metadata["token_usage"]["prompt_tokens_details"]["cached_tokens"]
print(f"Cached tokens: {cached}")

Cached tokens: 0


In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

# Use a cache key for repeated prompts
messages = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "I love programming."},
]

response = llm.invoke(
    messages,
    prompt_cache_key="translation-assistant-v1"
)

In [3]:
response

AIMessage(content="J'adore la programmation.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 26, 'total_tokens': 31, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_deacdd5f6f', 'id': 'chatcmpl-CxbfTBKDnQB42t9hezeVspqFWhbvj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bb837-3041-7280-9c8a-dfe9ac1ef284-0', usage_metadata={'input_tokens': 26, 'output_tokens': 5, 'total_tokens': 31, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Anthropic

### Langchain

In [14]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
)

# Static system prompt with cacheable block
system_msg = SystemMessage(
    content=[
        {
            "type": "text",
            "text": "You are a highly knowledgeable, reliable, and helpful AI assistant designed to support users across a wide range of domains, including but not limited to technology, computer science, data engineering, artificial intelligence, machine learning, mathematics, statistics, natural sciences, social sciences, history, philosophy, literature, arts, economics, business, and general knowledge."
        },
        {
            "type": "text",
            "text": long_static_system_prompt,
            "cache_control": {"type": "ephemeral"}  # Cache this block
        }
    ]
)

In [15]:
# First call: creates cache (cache_creation_input_tokens > 0)
messages1 = [system_msg, HumanMessage(content="First help me translate this sentence from English to French: I love programming.")]
resp1 = model.invoke(messages1)

In [22]:
pprint(resp1.model_dump())  # Shows cache_creation_input_tokens

{'additional_kwargs': {},
 'content': '# English to French Translation\n'
            '\n'
            '**"I love programming."**\n'
            '\n'
            'translates to:\n'
            '\n'
            '**"J\'aime la programmation."**\n'
            '\n'
            '---\n'
            '\n'
            '## Breakdown\n'
            '\n'
            '- **J\'aime** = "I love" (literally "I like," but *aimer* conveys '
            'strong affection and is the natural way to express love in '
            'French)\n'
            '- **la programmation** = "programming" (the noun form; *la* is '
            'the feminine article)\n'
            '\n'
            '---\n'
            '\n'
            '## Alternative phrasing\n'
            '\n'
            'If you wanted to express this more casually or emphasize the '
            'activity itself, you could also say:\n'
            '\n'
            '- **"J\'adore programmer."** — "I adore programming" (using the '
            'infinitive

In [23]:
print(resp1.content)  # Shows cache_creation_input_tokens

# English to French Translation

**"I love programming."**

translates to:

**"J'aime la programmation."**

---

## Breakdown

- **J'aime** = "I love" (literally "I like," but *aimer* conveys strong affection and is the natural way to express love in French)
- **la programmation** = "programming" (the noun form; *la* is the feminine article)

---

## Alternative phrasing

If you wanted to express this more casually or emphasize the activity itself, you could also say:

- **"J'adore programmer."** — "I adore programming" (using the infinitive verb form, which feels more dynamic)
- **"J'aime beaucoup programmer."** — "I really love programming" (adding *beaucoup* for emphasis)

The first translation I provided is the most standard and natural choice.


In [24]:
# Second call: hits cache (cache_read_input_tokens > 0)
messages2 = [system_msg, HumanMessage(content="Second help me translate this sentence from English to French: I love programming so much. I code every day.")]
resp2 = model.invoke(messages2)

In [25]:
pprint(resp2.model_dump())

{'additional_kwargs': {},
 'content': '# English to French Translation\n'
            '\n'
            '**I love programming so much. I code every day.**\n'
            '\n'
            '**French:**\n'
            "J'aime tellement la programmation. Je code tous les jours.\n"
            '\n'
            '---\n'
            '\n'
            '## Breakdown\n'
            '\n'
            '| English | French | Notes |\n'
            '|---------|--------|-------|\n'
            "| I love | J'aime | Standard present tense |\n"
            '| programming so much | tellement la programmation | "tellement" '
            '= so much; "la programmation" = programming (feminine noun) |\n'
            '| I code | Je code | Present tense; "coder" is commonly used in '
            'French tech contexts |\n'
            '| every day | tous les jours | Literally "all the days" |\n'
            '\n'
            '---\n'
            '\n'
            '## Alternative phrasing (slightly more natural):\n'
   

In [26]:
print(resp2.content)  # Shows cache_read_input_tokens, lower latency

# English to French Translation

**I love programming so much. I code every day.**

**French:**
J'aime tellement la programmation. Je code tous les jours.

---

## Breakdown

| English | French | Notes |
|---------|--------|-------|
| I love | J'aime | Standard present tense |
| programming so much | tellement la programmation | "tellement" = so much; "la programmation" = programming (feminine noun) |
| I code | Je code | Present tense; "coder" is commonly used in French tech contexts |
| every day | tous les jours | Literally "all the days" |

---

## Alternative phrasing (slightly more natural):

**J'adore la programmation. Je code chaque jour.**

- "J'adore" (I adore/love) is more emphatic and common in everyday French
- "chaque jour" (each day) is another natural way to say "every day"

Both versions are correct and natural—choose whichever feels right for your context!


### Direct API Call

In [34]:
import anthropic

client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    system=[
      {
        "type": "text",
        "text": "You are an AI assistant tasked with analyzing literary works. Your goal is to provide insightful commentary on themes, characters, and writing style.\n",
      },
      {
        "type": "text",
        "text": pride_prejudice_summary,
        "cache_control": {"type": "ephemeral"}
      }
    ],
    messages=[{"role": "user", "content": "Analyze the major themes in 'Pride and Prejudice'."}],
)
print(response.usage.model_dump_json())

# Call the model again with the same inputs up to the cache checkpoint
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    system=[
      {
        "type": "text",
        "text": "You are an AI assistant tasked with analyzing literary works. Your goal is to provide insightful commentary on themes, characters, and writing style.\n",
      },
      {
        "type": "text",
        "text": pride_prejudice_summary,
        "cache_control": {"type": "ephemeral"}
      }
    ],
    messages=[{"role": "user", "content": "Analyze the major themes in 'Pride and Prejudice'."}],
)
print(response.usage.model_dump_json())

{"cache_creation":{"ephemeral_1h_input_tokens":0,"ephemeral_5m_input_tokens":0},"cache_creation_input_tokens":0,"cache_read_input_tokens":0,"input_tokens":3576,"output_tokens":1024,"server_tool_use":null,"service_tier":"standard"}
{"cache_creation":{"ephemeral_1h_input_tokens":0,"ephemeral_5m_input_tokens":0},"cache_creation_input_tokens":0,"cache_read_input_tokens":0,"input_tokens":3576,"output_tokens":1024,"server_tool_use":null,"service_tier":"standard"}


In [33]:
# Call the model again with the same inputs up to the cache checkpoint
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    system=[
      {
        "type": "text",
        "text": "You are an AI assistant tasked with analyzing literary works. Your goal is to provide insightful commentary on themes, characters, and writing style.\n",
      },
      {
        "type": "text",
        "text": pride_prejudice_summary,
        "cache_control": {"type": "ephemeral"}
      }
    ],
    messages=[{"role": "user", "content": "Analyze the major themes in 'Pride and Prejudice'."}],
)
print(response.usage.model_dump_json())

{"cache_creation":{"ephemeral_1h_input_tokens":0,"ephemeral_5m_input_tokens":0},"cache_creation_input_tokens":0,"cache_read_input_tokens":0,"input_tokens":3576,"output_tokens":1024,"server_tool_use":null,"service_tier":"standard"}


Note (14/01/2026):

The caching is somehow not working properly even though I follow the exact instructions. Will try again later.